*Workflow*
Load train images -> Normalize -> Augment -> Train
Load test images -> Normalize -> Evaluate

In [ ]:
from PIL import Image
import numpy as np
import os
import tensorflow as tf
import random
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import gc



gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)


In [ ]:
def load_images(folder):
    images = []
    labels = []
    
    class_names = sorted(os.listdir(folder))
    
    for label, class_name in enumerate(class_names):
        class_path = os.path.join(folder, class_name)
        
        if not os.path.isdir(class_path):
            continue
        
        for file in os.listdir(class_path):
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                path = os.path.join(class_path, file)
                
                img = Image.open(path)
                img = np.array(img, dtype=np.float32)
                
                images.append(img)
                labels.append(label)
    
    images = np.array(images)
    labels = np.array(labels)
    
    return images, labels

In [ ]:
def augment_image(img):
    # Horizontal flip
    if random.random() > 0.5:
        img = np.fliplr(img)
    
    # Vertical flip
    if random.random() > 0.5:
        img = np.flipud(img)
    
    # Random rotation (90° steps for simplicity)
    k = random.randint(0, 3)
    img = np.rot90(img, k)
    
    return img

def augment_data(X_train, y_train):
    # Extract Ovarian Cancer (OC) class
    X_OC = X_train[y_train == 2]
    y_OC = y_train[y_train == 2]

    # Apply augmentation to training images
    X_OC_aug_1 = np.array([augment_image(img) for img in X_OC])
    X_OC_aug_2 = np.array([augment_image(img) for img in X_OC])
    X_OC_aug_3 = np.array([augment_image(img) for img in X_OC])
    X_OC_aug_4 = np.array([augment_image(img) for img in X_OC])

    # Combine original and augmented data
    X_train = np.concatenate([X_train, X_OC_aug_1, X_OC_aug_2, X_OC_aug_3, X_OC_aug_4])
    y_train = np.concatenate([y_train, y_OC, y_OC, y_OC, y_OC])

    print(X_train.min(), X_train.max()) # Check normalization
    unique, counts = np.unique(y_train, return_counts=True)
    print(dict(zip(unique, counts))) # Check class distribution after augmentation
    
    return X_train, y_train


In [ ]:
def compute_class_weights(y_train):
    class_weights = class_weight.compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )

    class_weights = dict(enumerate(class_weights))
    print(class_weights)
    
    return class_weights

In [ ]:
def build_and_train_model(X_train, y_train, X_test, y_test, class_weights):
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(224,224,1)),

        tf.keras.layers.Conv2D(32, (3,3), strides=1, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

        tf.keras.layers.Conv2D(64, (5,5), strides=1, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

        tf.keras.layers.Conv2D(128, (3,3), strides=1, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),

        tf.keras.layers.Flatten(), 
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(4, activation='softmax')  # Multi-Class classification (4 classes)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=25,
        batch_size=16,
        class_weight=class_weights,
        verbose=1
    )

    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('Model Accuracy')
    plt.legend(['Train', 'Validation'])
    plt.show()

    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.legend(['Train', 'Validation'])
    plt.show()

    return model, history

In [ ]:
# One-Vs-Rest (OvR) Evaluation
def evaluate_model(model, X_test, y_test):
    y_probs = model.predict(X_test)
    y_pred = np.argmax(y_probs, axis=1)

    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    y_test_bin = label_binarize(y_test, classes=[0,1,2,3])
    auc_val = roc_auc_score(y_test_bin, y_probs, average='macro')

    cm = confusion_matrix(y_test, y_pred)

    print("Accuracy:", acc)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)
    print("AUC:", auc_val)
    print("Confusion Matrix:\n", cm)
    return acc, precision, recall, f1, auc_val

In [ ]:
def save_background_data(X_train, y_train):
    background_size_per_class = 10
    background_indices = []

    for class_label in range(4):
        class_indices = np.where(y_train == class_label)[0]
        selected = np.random.choice(class_indices, background_size_per_class, replace=False)
        background_indices.extend(selected)

    background_data = X_train[background_indices]
    np.save("background_data.npy", background_data)
    print("Background data saved for SHAP explanations.")

In [ ]:
best_recall = -1
best_fold = None
best_model_path = "aocdml.keras"


# Loop through each fold and train/evaluate the model
all_fold_results = []
all_y_true = []
all_y_prob = []
for fold in range(1, 6):
    print(f"\n=========== FOLD {fold} ===========")
    tf.keras.backend.clear_session()
    train_dir = rf"cross_val_folds/fold_{fold}/train"
    test_dir = rf"cross_val_folds/fold_{fold}/test"

    # Load images and labels
    X_train, y_train = load_images(train_dir)
    X_test, y_test = load_images(test_dir)

    # Normalize image data
    X_train = X_train / 255.0
    X_test = X_test / 255.0

    # Add channel dimension
    X_train = np.expand_dims(X_train, axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)

    # Augment training data
    X_train, y_train = augment_data(X_train, y_train)
   
    # Compute class weights
    class_weights = compute_class_weights(y_train)
   
    # Build and compile model
    model, history = build_and_train_model(X_train, y_train, X_test, y_test, class_weights)
   
    # Evaluate and save model
    acc, precision, recall, f1, auc_val = evaluate_model(model, X_test, y_test)
    if recall > best_recall:
        print(f"New best model found at fold {fold} (Recall = {recall:.4f})")
        best_recall = recall
        best_fold = fold
        model.save(best_model_path)
        save_background_data(X_train, y_train)
    all_fold_results.append([acc, precision, recall, f1, auc_val])
   
    # Store true labels and predicted probabilities for ROC curve analysis
    y_prob = model.predict(X_test)
    all_y_true.append(y_test)
    all_y_prob.append(y_prob)

    del X_train, y_train, X_test, y_test, model, history
    gc.collect()
    
    # Optional: reset GPU memory stats
    try:
        tf.config.experimental.reset_memory_stats('GPU:0')
    except:
        pass
   
# Print average results across folds
all_fold_results = np.array(all_fold_results)
y_true_all = np.concatenate(all_y_true)
y_prob_all = np.concatenate(all_y_prob)
print("\n===== FINAL CROSS VALIDATION RESULTS =====")
print("Mean Accuracy:", all_fold_results[:,0].mean())
print("Mean Precision:", all_fold_results[:,1].mean())
print("Mean Recall:", all_fold_results[:,2].mean())
print("Mean F1:", all_fold_results[:,3].mean())
print("Mean AUC:", all_fold_results[:,4].mean())
print(f"\nBest model was from Fold {best_fold} with recall = {best_recall:.4f}")
print(f"Saved as: {best_model_path}")


In [ ]:
n_classes = 4
y_true_bin = label_binarize(y_true_all, classes=[0,1,2,3])

plt.figure()

for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob_all[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"Class {i} (AUC = {roc_auc:.3f})")

plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Cross-Validated ROC Curve")
plt.legend()
plt.show()

In [ ]:
# Training Accuracy and Validation Accuracy Plotting
train_acc_folds = [
    [0.6667, 0.8418, 0.9078, 0.9291, 0.9426, 0.9511, 0.9631, 0.9695, 0.9773, 0.9816, 0.9823, 0.9908, 0.9865, 0.995, 0.9887, 0.9887, 0.9922, 0.9908, 0.9972, 0.9979, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986],
    [0.6773, 0.8511, 0.9064, 0.9333, 0.9411, 0.9546, 0.9645, 0.9709, 0.9787, 0.9858, 0.9879, 0.9936, 0.9936, 0.9929, 0.995, 0.9957, 0.9957, 0.9979, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    [0.6652, 0.8631, 0.9057, 0.939, 0.9468, 0.9589, 0.9723, 0.9759, 0.9865, 0.9901, 0.9922, 0.9936, 0.9979, 0.9979, 0.9979, 0.9979, 0.9972, 0.9957, 0.9929, 0.9986, 0.9986, 0.9979, 0.9986, 0.9986, 0.9986],
    [0.6459, 0.8516, 0.9067, 0.9244, 0.9364, 0.9527, 0.9647, 0.9739, 0.9845, 0.9887, 0.9915, 0.9936, 0.9972, 0.9986, 0.9979, 0.9979, 0.9986, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    [0.6424, 0.8459, 0.8947, 0.9223, 0.9428, 0.9555, 0.976, 0.9788, 0.9859, 0.9859, 0.9922, 0.9943, 0.9951, 0.9965, 0.9979, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986, 0.9986],
]

val_acc_folds = [
    [0.8714, 0.8971, 0.9068, 0.9132, 0.9196, 0.926, 0.9293, 0.9293, 0.9325, 0.9228, 0.9293, 0.9357, 0.9389, 0.9389, 0.9389, 0.9325, 0.9293, 0.9293, 0.9293, 0.9293, 0.9357, 0.9003, 0.9357, 0.9357, 0.9389],
    [0.8682, 0.9068, 0.9164, 0.9228, 0.9228, 0.926, 0.9357, 0.9357, 0.9357, 0.9389, 0.9357, 0.9164, 0.9003, 0.8939, 0.926, 0.9357, 0.9389, 0.9389, 0.9453, 0.9421, 0.9421, 0.9421, 0.9421, 0.9421, 0.9486],
    [0.9035, 0.9325, 0.9293, 0.9421, 0.9389, 0.9389, 0.926, 0.9293, 0.9357, 0.9389, 0.9389, 0.9486, 0.9486, 0.9486, 0.9518, 0.9357, 0.9389, 0.9453, 0.955, 0.9421, 0.9486, 0.9453, 0.9453, 0.9486, 0.9421],
    [0.8419, 0.8774, 0.871, 0.8742, 0.8806, 0.8968, 0.8968, 0.9226, 0.9065, 0.9065, 0.9129, 0.9065, 0.9032, 0.8871, 0.8871, 0.9194, 0.9258, 0.9032, 0.9194, 0.929, 0.9258, 0.9258, 0.9258, 0.929, 0.929],
    [0.8323, 0.8452, 0.8581, 0.8839, 0.8903, 0.9097, 0.9258, 0.929, 0.9258, 0.9194, 0.9323, 0.929, 0.9323, 0.9419, 0.9419, 0.9484, 0.9452, 0.9484, 0.9452, 0.9484, 0.9387, 0.9387, 0.9419, 0.9419, 0.9452],
]


# Convert to numpy arrays for easy calculation
train_acc_folds = np.array(train_acc_folds)
val_acc_folds = np.array(val_acc_folds)

# Compute mean and std per epoch
train_mean = train_acc_folds.mean(axis=0)
train_std = train_acc_folds.std(axis=0)
val_mean = val_acc_folds.mean(axis=0)
val_std = val_acc_folds.std(axis=0)

epochs = np.arange(1, 26)

# Plot with shaded std
plt.figure(figsize=(10,6))
plt.plot(epochs, train_mean, label='Training Accuracy', color='blue')
plt.fill_between(epochs, train_mean - train_std, train_mean + train_std, color='blue', alpha=0.2)
plt.plot(epochs, val_mean, label='Validation Accuracy', color='orange')
plt.fill_between(epochs, val_mean - val_std, val_mean + val_std, color='orange', alpha=0.2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training & Validation Accuracy Across 5 Folds')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Training Loss and Validation Loss Plotting
train_loss_folds = [
    [0.9093, 0.4571, 0.2997, 0.2256, 0.1854, 0.154, 0.131, 0.1073, 0.0862, 0.0761, 0.0625, 0.0455, 0.0448, 0.0332, 0.0343, 0.0366, 0.0318, 0.0353, 0.0165, 0.0104, 0.0074, 0.0077, 0.0088, 0.0061, 0.0068],
    [0.9016, 0.4466, 0.2981, 0.2228, 0.1832, 0.1444, 0.115, 0.0903, 0.0682, 0.0586, 0.0393, 0.0323, 0.0298, 0.0261, 0.0207, 0.0141, 0.0168, 0.013, 0.0065, 0.0046, 0.0038, 0.0032, 0.0025, 0.0018, 0.0016],
    [0.9169, 0.4384, 0.2972, 0.2245, 0.1783, 0.1412, 0.1097, 0.089, 0.0617, 0.0461, 0.0442, 0.0278, 0.018, 0.0143, 0.0162, 0.0147, 0.0151, 0.0187, 0.0282, 0.0119, 0.0119, 0.0121, 0.0099, 0.0088, 0.0105],
    [0.8931, 0.452, 0.2963, 0.2272, 0.1874, 0.1544, 0.1182, 0.0946, 0.0711, 0.0509, 0.0374, 0.0279, 0.0179, 0.0162, 0.0149, 0.0127, 0.0103, 0.0049, 0.0047, 0.0036, 0.0024, 0.0026, 0.0021, 0.0012, 0.0013],
    [0.9087, 0.4787, 0.3271, 0.2424, 0.1797, 0.1403, 0.0973, 0.0783, 0.0588, 0.0521, 0.0378, 0.0303, 0.0254, 0.018, 0.0166, 0.0112, 0.0111, 0.0114, 0.011, 0.0108, 0.0095, 0.0078, 0.0077, 0.0086, 0.0063],
]

val_loss_folds = [
    [0.4362, 0.3338, 0.2962, 0.279, 0.2743, 0.2838, 0.2807, 0.2868, 0.2997, 0.3102, 0.3086, 0.3577, 0.3741, 0.3894, 0.4208, 0.4037, 0.3843, 0.3858, 0.3815, 0.4091, 0.4198, 0.4837, 0.4383, 0.4516, 0.4575],
    [0.4282, 0.3325, 0.2861, 0.2627, 0.2481, 0.2416, 0.2295, 0.229, 0.2382, 0.23, 0.2427, 0.2973, 0.3772, 0.3913, 0.2704, 0.2505, 0.2541, 0.2538, 0.2629, 0.2747, 0.2773, 0.2795, 0.2887, 0.3025, 0.3051],
    [0.4717, 0.3326, 0.308, 0.31, 0.2961, 0.284, 0.2887, 0.2628, 0.267, 0.2771, 0.2691, 0.3042, 0.3359, 0.3928, 0.3684, 0.3098, 0.3078, 0.3562, 0.3524, 0.3321, 0.339, 0.3394, 0.3469, 0.3547, 0.3591],
    [0.3265, 0.2304, 0.1986, 0.2203, 0.2211, 0.2212, 0.2264, 0.2138, 0.2199, 0.2299, 0.2314, 0.2534, 0.2645, 0.2987, 0.2885, 0.241, 0.2275, 0.271, 0.2738, 0.2545, 0.2677, 0.2739, 0.2713, 0.2805, 0.2802],
    [0.4866, 0.3669, 0.3549, 0.278, 0.279, 0.2531, 0.2364, 0.2262, 0.2432, 0.251, 0.2323, 0.2092, 0.2147, 0.201, 0.1998, 0.2012, 0.216, 0.2094, 0.209, 0.2198, 0.2329, 0.2438, 0.2591, 0.2627, 0.2523],
]

# Convert to numpy arrays for easy calculation
train_loss_folds = np.array(train_loss_folds)
val_loss_folds = np.array(val_loss_folds)

# Compute mean and std per epoch
train_mean = train_loss_folds.mean(axis=0)
train_std = train_loss_folds.std(axis=0)
val_mean = val_loss_folds.mean(axis=0)
val_std = val_loss_folds.std(axis=0)

epochs = np.arange(1, 26)

# Plot with shaded std
plt.figure(figsize=(10,6))
plt.plot(epochs, train_mean, label='Training Loss', color='blue')
plt.fill_between(epochs, train_mean - train_std, train_mean + train_std, color='blue', alpha=0.2)
plt.plot(epochs, val_mean, label='Validation Loss', color='orange')
plt.fill_between(epochs, val_mean - val_std, val_mean + val_std, color='orange', alpha=0.2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training & Validation Loss Across 5 Folds')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Test Performance Metrics Plotting

metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
values = [0.9408, 0.8101, 0.8141, 0.8115, 0.9720]

plt.bar(metrics, values, color=['skyblue', 'orange', 'green', 'red', 'purple'])
plt.ylim(0, 1)
plt.title("Average Test Performance Metrics")
plt.show()


In [ ]:
# Confusion Matrix Plotting
cm = np.array([[467,   3,  23,   7],
                [  4, 494,   2,   0],
                [ 32,   1,  20,   0],
                [  9,   0,  11, 480]])

class_names = ['BOT', 'NO', 'OC', 'UF']

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Cross Validation Accuracy Plotting

folds = ['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5']
accuracy = [0.9389, 0.9485, 0.9421, 0.9290, 0.9452]

plt.bar(folds, accuracy, color=['red', 'yellow', 'green', 'darkblue', 'purple'])
plt.ylim(0, 1)
plt.title("CNN Accuracy Across 5 Folds")
plt.show()
